# 04 — Exploratory Data Analysis (EDA)

## Purpose

Explore the cleaned fleet data before statistical modelling.

### Analytical workflow

**Question → inspection → visualization → finding → business meaning**

This notebook uses the ** processed datasets** created by the cleaning phase.

### Main datasets

- `trips_clean.csv` — trip-level operations
- `trucks_clean.csv` — truck-level KPIs
- `drivers_clean.csv` — driver-level KPIs
- `maintenance_clean.csv` — maintenance history
- `fuel_purchases_clean.csv` — fuel transactions

> The data is synthetic and represents a simulated Zimbabwean trucking operation.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Keep the project path relative so the notebook works when the project is moved.
PROJECT = Path.cwd().parent
DATA_DIR = PROJECT / "data" / "processed"

print("Project:", PROJECT)
print("Processed data:", DATA_DIR)
print("Processed folder exists:", DATA_DIR.exists())


In [ ]:
# Load the processed datasets.

trips = pd.read_csv(DATA_DIR / "trips_clean.csv")
trucks = pd.read_csv(DATA_DIR / "trucks_clean.csv")
drivers = pd.read_csv(DATA_DIR / "drivers_clean.csv")
maintenance = pd.read_csv(DATA_DIR / "maintenance_clean.csv")
fuel = pd.read_csv(DATA_DIR / "fuel_purchases_clean.csv")

# Convert date columns to datetime.
trips["trip_date"] = pd.to_datetime(trips["trip_date"])
maintenance["maintenance_date"] = pd.to_datetime(maintenance["maintenance_date"])
fuel["purchase_date"] = pd.to_datetime(fuel["purchase_date"])

print(f"Trips: {trips.shape}")
print(f"Trucks: {trucks.shape}")
print(f"Drivers: {drivers.shape}")
print(f"Maintenance records: {maintenance.shape}")
print(f"Fuel purchases: {fuel.shape}")


## 1. Data-quality overview

Before interpreting patterns, check:

- number of rows and columns
- missing values
- duplicate records
- data types
- basic ranges

This prevents us from interpreting a data-quality problem as a business finding.


In [ ]:
# Missing values by dataset.

datasets = {
    "trips": trips,
    "trucks": trucks,
    "drivers": drivers,
    "maintenance": maintenance,
    "fuel": fuel,
}

for name, df in datasets.items():
    missing = df.isna().sum().sum()
    duplicates = df.duplicated().sum()
    print(f"{name:12} | rows={len(df):4,} | columns={df.shape[1]:2} | "
          f"missing={missing:3} | duplicates={duplicates:3}")


In [ ]:
# Inspect the main trip-level numeric variables.

trips[
    [
        "distance_km",
        "cargo_weight_tons",
        "fuel_consumed_litres",
        "revenue",
        "operating_cost",
        "profit",
        "duration_hours",
        "delay_minutes",
        "fuel_efficiency_km_per_l",
        "profit_margin_pct",
        "cost_per_km",
    ]
].describe().round(2)


**Interpretation:** The descriptive table gives the central tendency and spread of the main operational measures. Pay particular attention to variables such as distance, fuel consumption, revenue, cost and profit because they drive later analysis.


## 2. Fleet composition

**Question:** What does the simulated fleet look like?

We examine truck age, capacity and model mix.


In [ ]:
print("Truck models:")
display(trucks["model"].value_counts())

print("\nTruck age summary:")
display(trucks["age"].describe().round(2))

print("\nCapacity summary:")
display(trucks["capacity_tons"].describe().round(2))


In [ ]:
# Truck age distribution.

plt.figure(figsize=(8, 5))
plt.hist(trucks["age"], bins=range(int(trucks["age"].min()), int(trucks["age"].max()) + 2), edgecolor="black")
plt.title("Truck Age Distribution")
plt.xlabel("Truck age (years)")
plt.ylabel("Number of trucks")
plt.tight_layout()
plt.show()


**Business meaning:** Fleet age is important because vehicle age can affect maintenance requirements, downtime and operating economics.


## 3. Trip volume and route mix

**Question:** Which routes and destinations account for the most activity?


In [ ]:
route_summary = (
    trips.groupby("route")
    .agg(
        trips=("trip_id", "count"),
        distance_km=("distance_km", "sum"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
    )
    .sort_values("trips", ascending=False)
)

route_summary.round(2)


In [ ]:
# Top routes by number of trips.

top_routes = route_summary.head(10).sort_values("trips")

plt.figure(figsize=(9, 5))
plt.barh(top_routes.index, top_routes["trips"])
plt.title("Top Routes by Trip Volume")
plt.xlabel("Number of trips")
plt.ylabel("Route")
plt.tight_layout()
plt.show()


**Interpretation:** Trip volume shows where the fleet is most heavily utilized. High-volume routes should not automatically be treated as the most profitable; volume and profitability need to be compared.


## 4. Revenue, cost and profitability

**Question:** How is the fleet performing financially at trip level?


In [ ]:
financial_summary = trips[
    ["revenue", "operating_cost", "profit", "profit_margin_pct", "cost_per_km"]
].describe().round(2)

financial_summary


In [ ]:
# Revenue versus operating cost by trip.

sample = trips.sort_values("revenue").reset_index(drop=True)

plt.figure(figsize=(10, 5))
plt.plot(sample["revenue"].values, label="Revenue")
plt.plot(sample["operating_cost"].values, label="Operating cost")
plt.title("Trip Revenue and Operating Cost")
plt.xlabel("Trips ordered by revenue")
plt.ylabel("Amount ($)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Profit distribution.

plt.figure(figsize=(8, 5))
plt.hist(trips["profit"], bins=30, edgecolor="black")
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Trip Profit Distribution")
plt.xlabel("Profit ($)")
plt.ylabel("Number of trips")
plt.tight_layout()
plt.show()


**Interpretation:** The zero line separates profitable from loss-making trips. This is important because the data intentionally contains both outcomes, allowing profitability to be investigated as a real business problem.


## 5. Fuel consumption and efficiency

**Question:** What factors appear to be associated with fuel consumption and fuel efficiency?

Important distinction:

- `fuel_efficiency_km_per_l` is useful for **descriptive analysis**.
- It must not be used as a predictor for a model whose target is `fuel_consumed_litres`, because efficiency is calculated from distance and fuel consumed.


In [ ]:
fuel_summary = trips[
    ["distance_km", "cargo_weight_tons", "fuel_consumed_litres",
     "fuel_efficiency_km_per_l", "fuel_cost"]
].describe().round(2)

fuel_summary


In [ ]:
# Distance versus fuel consumed.

plt.figure(figsize=(8, 5))
plt.scatter(
    trips["distance_km"],
    trips["fuel_consumed_litres"],
    alpha=0.5
)
plt.title("Distance vs Fuel Consumed")
plt.xlabel("Distance (km)")
plt.ylabel("Fuel consumed (litres)")
plt.tight_layout()
plt.show()


In [ ]:
# Cargo weight versus fuel consumed.

plt.figure(figsize=(8, 5))
plt.scatter(
    trips["cargo_weight_tons"],
    trips["fuel_consumed_litres"],
    alpha=0.5
)
plt.title("Cargo Weight vs Fuel Consumed")
plt.xlabel("Cargo weight (tons)")
plt.ylabel("Fuel consumed (litres)")
plt.tight_layout()
plt.show()


**Interpretation:** These plots help us visually inspect whether longer trips and heavier loads are associated with higher fuel consumption. We will quantify relationships later in the statistical-analysis phase.


## 6. Delay analysis

**Question:** Where are delays concentrated?

We examine overall delay rate, delay duration, routes, road conditions and border crossings.


In [ ]:
delay_rate = trips["has_delay"].mean() * 100

print(f"Overall delay rate: {delay_rate:.2f}%")
print(f"Average delay among all trips: {trips['delay_minutes'].mean():.2f} minutes")
print(f"Average delay when delayed: {trips.loc[trips['has_delay'] == 1, 'delay_minutes'].mean():.2f} minutes")


In [ ]:
delay_by_route = (
    trips.groupby("route")
    .agg(
        trips=("trip_id", "count"),
        delayed_trips=("has_delay", "sum"),
        delay_rate_pct=("has_delay", "mean"),
        avg_delay_minutes=("delay_minutes", "mean"),
    )
)

delay_by_route["delay_rate_pct"] *= 100
delay_by_route.sort_values("delay_rate_pct", ascending=False).round(2).head(10)


In [ ]:
# Delay rate by road condition.

road_delay = (
    trips.groupby("road_condition")["has_delay"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(road_delay.index, road_delay.values)
plt.title("Delay Rate by Road Condition")
plt.xlabel("Road condition")
plt.ylabel("Delay rate (%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Delay rate for border versus non-border trips.

border_delay = (
    trips.groupby("border_crossing_flag")["has_delay"]
    .mean()
    .mul(100)
)

border_delay.index = ["No border crossing", "Border crossing"]

plt.figure(figsize=(7, 5))
plt.bar(border_delay.index, border_delay.values)
plt.title("Delay Rate by Border-Crossing Status")
plt.ylabel("Delay rate (%)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


**Interpretation:** Delay analysis identifies operational conditions that deserve further investigation. EDA shows patterns; the next statistical phase will test whether selected relationships are statistically supported.


## 7. Maintenance and vehicle age

**Question:** Does maintenance burden appear to increase with vehicle age and mileage?


In [ ]:
maintenance_by_truck = (
    trucks[
        ["truck_id", "age", "mileage_km", "maintenance_count",
         "total_maintenance_cost", "total_downtime_days"]
    ]
    .sort_values("total_maintenance_cost", ascending=False)
)

maintenance_by_truck.head(10).round(2)


In [ ]:
# Truck age vs maintenance count.

plt.figure(figsize=(8, 5))
plt.scatter(trucks["age"], trucks["maintenance_count"], alpha=0.7)
plt.title("Truck Age vs Maintenance Count")
plt.xlabel("Truck age (years)")
plt.ylabel("Maintenance events")
plt.tight_layout()
plt.show()


In [ ]:
# Mileage vs maintenance count.

plt.figure(figsize=(8, 5))
plt.scatter(trucks["mileage_km"], trucks["maintenance_count"], alpha=0.7)
plt.title("Mileage vs Maintenance Count")
plt.xlabel("Mileage (km)")
plt.ylabel("Maintenance events")
plt.tight_layout()
plt.show()


**Interpretation:** These plots are designed to reveal the maintenance relationships visually. Statistical tests will determine the strength and significance of these relationships in the next phase.


## 8. Driver performance overview

**Question:** How does driver performance vary across the fleet?

We examine trips, skill score, delay rate and profit without treating any single metric as a complete measure of driver quality.


In [ ]:
driver_summary = drivers[
    [
        "driver_id",
        "full_name",
        "total_trips",
        "driver_skill_score",
        "delay_rate_pct",
        "total_profit",
        "avg_fuel_efficiency_km_per_l",
    ]
].sort_values("total_trips", ascending=False)

driver_summary.head(10).round(2)


In [ ]:
# Skill score distribution.

plt.figure(figsize=(8, 5))
plt.hist(drivers["driver_skill_score"], bins=12, edgecolor="black")
plt.title("Driver Skill Score Distribution")
plt.xlabel("Skill score")
plt.ylabel("Number of drivers")
plt.tight_layout()
plt.show()


**Interpretation:** Driver-level EDA identifies variation that can later be investigated alongside route mix, trip volume and delay behaviour.


## 9. Maintenance category and severity

**Question:** What types of maintenance occur most frequently, and which categories are most expensive?


In [ ]:
maintenance_category = (
    maintenance.groupby("category")
    .agg(
        events=("maintenance_id", "count"),
        total_cost=("cost", "sum"),
        avg_cost=("cost", "mean"),
        total_downtime=("downtime_days", "sum"),
    )
    .sort_values("total_cost", ascending=False)
)

maintenance_category.round(2)


In [ ]:
# Maintenance events by category.

category_counts = maintenance["category"].value_counts()

plt.figure(figsize=(8, 5))
plt.bar(category_counts.index, category_counts.values)
plt.title("Maintenance Events by Category")
plt.xlabel("Maintenance category")
plt.ylabel("Number of events")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


**Interpretation:** Frequency and cost tell different stories. A maintenance category may be frequent but inexpensive, or less frequent but financially significant.


## 10. Fuel purchasing analysis

**Question:** Where and when is the fleet spending money on fuel?


In [ ]:
fuel_by_location = (
    fuel.groupby("location")
    .agg(
        purchases=("fuel_id", "count"),
        litres=("litres", "sum"),
        total_cost=("total_cost", "sum"),
        avg_price_per_litre=("price_per_litre", "mean"),
    )
    .sort_values("total_cost", ascending=False)
)

fuel_by_location.round(2)


In [ ]:
# Fuel price distribution.

plt.figure(figsize=(8, 5))
plt.hist(fuel["price_per_litre"], bins=20, edgecolor="black")
plt.title("Fuel Price per Litre")
plt.xlabel("Price per litre ($)")
plt.ylabel("Number of purchases")
plt.tight_layout()
plt.show()


**Interpretation:** Fuel expenditure can be decomposed into litres purchased and price per litre. This distinction is useful when investigating cost-control opportunities.


## 11. Correlation overview

**Question:** Which numerical variables appear related?

Correlation is exploratory evidence, not proof of causation. Statistical tests will be used in the next phase.


In [ ]:
corr_cols = [
    "distance_km",
    "cargo_weight_tons",
    "fuel_consumed_litres",
    "revenue",
    "operating_cost",
    "profit",
    "duration_hours",
    "delay_minutes",
    "loading_time_hours",
    "road_condition_score",
    "weather_risk_score",
    "route_risk_score",
    "fuel_efficiency_km_per_l",
    "profit_margin_pct",
    "cost_per_km",
]

corr = trips[corr_cols].corr()

display(corr.round(2))


In [ ]:
# Display the strongest absolute correlations with profit.

profit_corr = (
    corr["profit"]
    .drop("profit")
    .abs()
    .sort_values(ascending=False)
)

profit_corr.head(10)


**Interpretation:** The correlation table helps prioritize relationships for formal testing and modelling. A strong correlation does not by itself establish causation.


## 12. Key EDA findings

Use the results above to record findings in this structure:

| Finding | Evidence | Business meaning |
|---|---|---|
| Example: older trucks show more maintenance events | Age vs maintenance plot | Supports further investigation of preventive maintenance |
| Example: some routes have higher delay rates | Route delay table | Indicates routes requiring operational review |
| Example: longer trips consume more fuel | Distance vs fuel plot | Supports fuel-consumption modelling |

### Important

Do not fill this section with assumptions. Write the final findings only after reviewing the actual outputs generated by your run.

## What EDA has established

EDA has helped us:

1. Understand the structure of the fleet data.
2. Inspect distributions and potential outliers.
3. Identify operational patterns worth testing.
4. Separate descriptive findings from statistical evidence.
5. Identify candidate variables for machine-learning models.

### Next phase

**05 — Statistical Analysis **

There we will formally test selected relationships identified during EDA.
